In [0]:
dbutils.widgets.dropdown(name = 'Environment', defaultValue = 'dev', choices = ['dev','qa','prd'], label = 'Environment')
Env = dbutils.widgets.get("Environment")

In [0]:
goldTableName = f"saleslake_{Env}.gold_{Env}.refineddiscount"
print(goldTableName)
silverTableName = f"saleslake_{Env}.silver_{Env}.cleaneddiscount"
print(silverTableName)

In [0]:
spark.sql(f"""MERGE INTO {goldTableName} AS tgt
USING (
    SELECT * FROM {silverTableName}
    WHERE ingest_ts >
    (
        SELECT COALESCE(
            MAX(last_updt_ts),
            to_timestamp('1990-01-01','yyyy-MM-dd')
        )
        FROM {goldTableName}
    )
) AS src

ON tgt.discount_id = src.discount_id

WHEN MATCHED  AND (
       tgt.discount_code <> src.discount_code
    OR tgt.discount_name <> src.discount_name
    OR tgt.discount_type <> src.discount_type
    OR tgt.discount_value <> src.discount_value
    OR tgt.min_purchase_amount <> src.min_purchase_amount
    OR tgt.max_discount_amount <> src.max_discount_amount
    OR tgt.valid_from <> src.valid_from
)
THEN UPDATE SET
    tgt.discount_code = src.discount_code,
    tgt.discount_name = src.discount_name,
    tgt.discount_type = src.discount_type,
    tgt.discount_value = src.discount_value,
    tgt.min_purchase_amount = src.min_purchase_amount,
    tgt.max_discount_amount = src.max_discount_amount,
    tgt.valid_from = src.valid_from,
    tgt.last_updt_ts = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
INSERT (
    discount_id,
    discount_code,
    discount_name,
    discount_type,
    discount_value,
    min_purchase_amount,
    max_discount_amount,
    valid_from,
    initial_load_ts,
    last_updt_ts
)
VALUES (
    src.discount_id,
    src.discount_code,
    src.discount_name,
    src.discount_type,
    src.discount_value,
    src.min_purchase_amount,
    src.max_discount_amount,
    src.valid_from,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP()
)
""")

In [0]:
%sql
--SELECT * FROM saleslake_dev.gold_dev.refineddiscount;